# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/summayashaikh079-stack/flyrank-ml-week1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Finding chosen — "What Predicts Health?" (ML Appendix, Random Forest feature importance for health score).
The paper reports Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the top predictors of health_score, from a holdout-tested Random Forest.

My methodology question: Where does the label come from? The paper itself flags this honestly — health_score is a composite metric built from Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts). Three of the four top predicted-from features (Average Position, Impressions, Scroll Depth) are also direct ingredients of the label itself. That's the label-derived-feature pattern: the model isn't discovering an external cause of health, it's partially reconstructing an arithmetic formula it was already fed pieces of. My question, asked the way I'd want mine asked: would this ranking hold if health score were computed WITHOUT position/impressions/scroll — or is the 43%/32%/15% split mostly restating the scoring formula back to us?

Finding chosen — "What Predicts Growth?" (ML Appendix, Logistic Regression, 71% holdout accuracy).
Content age, days-since-update, and days-visible are reported as the strongest signals separating growing from declining pages.

My methodology question: The growth/decline label is built from 30-day-vs-previous-30-day impression change — independent of the predictors used (age, freshness, visibility), so this pairing looks cleaner than the health-score case. But the paper doesn't say whether the 80/20 holdout split is a random row split or grouped by brand. With 57 brands in the portfolio, a random split could let the model see 90% of a brand's pages in training and the other 10% in test — learning brand-specific patterns rather than a generalizable signal. My question, framed constructively: was the split brand-grouped, and if not, would the 71% accuracy hold on brands the model has never seen at all?

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
findings = {
    "What Predicts Health? (page 27, ML Appendix - Feature Importance)":
        "Label = health_score = Impressions(30) + Position(30) + CTR(20) + Scroll(20). "
        "Top predictors: Average Position 43%, Impressions 32%, Scroll Depth 15%.",
    "What Predicts Growth? (page 29, ML Appendix - Growth & Classification)":
        "Label = 30d-vs-prev-30d impression trend (up/down). Logistic regression, 71% holdout accuracy. "
        "Split grouping (random vs brand-grouped) not stated in the paper.",
}
for k, v in findings.items():
    print(f"- {k}\n  {v}\n")

- What Predicts Health? (page 27, ML Appendix - Feature Importance)
  Label = health_score = Impressions(30) + Position(30) + CTR(20) + Scroll(20). Top predictors: Average Position 43%, Impressions 32%, Scroll Depth 15%.

- What Predicts Growth? (page 29, ML Appendix - Growth & Classification)
  Label = 30d-vs-prev-30d impression trend (up/down). Logistic regression, 71% holdout accuracy. Split grouping (random vs brand-grouped) not stated in the paper.



## 2. My model under an honest split (before/after)

My Week-5 model (is_declining_label, Random Forest) was already trained under a client-grouped split — good practice carried over. To make the "before/after" honest and visible, I re-ran the same model and same features under a naive random row split first (same client can appear in both train and test), then under the client-grouped split (a client appears in only one side). Same data, same features, same Precision@50 metric, side by side.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
import os, subprocess
if not os.path.isdir("flyrank-ml-week1"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/summayashaikh079-stack/flyrank-ml-week1"], check=True)
os.chdir("flyrank-ml-week1")

import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

NUMERIC_FILL = ["search_volume","competition","cpc","word_count","char_count","impressions_90d","clicks_90d",
    "pageviews_90d","sessions_90d","users_90d","engaged_sessions_90d","ai_sessions_90d","scroll_events_90d",
    "days_with_impressions","days_with_sessions","impressions_last_30d","clicks_last_30d","sessions_last_30d",
    "impressions_prev_30d","clicks_prev_30d","sessions_prev_30d","content_age_days","age_tier_order",
    "days_since_last_update","ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct","trend_pct"]
for c in NUMERIC_FILL:
    df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0)

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

MODEL_NUMERIC = ["search_volume","competition","cpc","word_count","char_count","log_impressions_90d",
    "log_clicks_90d","log_sessions_90d","log_ai_sessions_90d","days_with_impressions","days_with_sessions",
    "content_age_days","days_since_last_update","ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct"]
MODEL_CAT = ["competition_level","content_type","main_intent","age_tier","freshness_tier",
    "word_count_tier","impression_tier","position_tier"]

num_frame = df[MODEL_NUMERIC].apply(pd.to_numeric, errors="coerce").fillna(0)
cat_frame = pd.get_dummies(df[MODEL_CAT].fillna("unknown").astype(str), prefix=MODEL_CAT, dtype=float)
X = pd.concat([num_frame, cat_frame], axis=1)
y = df["is_declining_label"].astype(int)
groups = df["client_id"]

def precision_at_k(y_true, scores, k):
    f = pd.DataFrame({"y": y_true, "score": scores}).sort_values("score", ascending=False).head(k)
    return f["y"].mean()

def run_rf(Xtr, Xte, ytr, yte):
    rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
                                 n_estimators=200, n_jobs=-1, random_state=42)
    rf.fit(Xtr, ytr)
    proba = rf.predict_proba(Xte)[:, 1]
    return roc_auc_score(yte, proba), precision_at_k(yte, proba, 50), yte.mean()

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
auc_before, p50_before, base_before = run_rf(Xtr, Xte, ytr, yte)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
auc_after, p50_after, base_after = run_rf(X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx])

result = pd.DataFrame([
    {"split": "BEFORE - random row split", "roc_auc": round(auc_before,3), "precision_at_50": round(p50_before,3), "base_rate": round(base_before,3)},
    {"split": "AFTER - client-grouped split", "roc_auc": round(auc_after,3), "precision_at_50": round(p50_after,3), "base_rate": round(base_after,3)},
])
print(result.to_string(index=False))
print(f"\nClient overlap between train/test in the grouped split: {len(overlap)} (0 = honest)")

                       split  roc_auc  precision_at_50  base_rate
   BEFORE - random row split    0.758             0.90      0.542
AFTER - client-grouped split    0.610             0.54      0.511

Client overlap between train/test in the grouped split: 0 (0 = honest)


## 3. Leakage audit
Checklist against my final feature set (MODEL_NUMERIC + MODEL_CATEGORICAL from scripts/ml_utils.py):
- Timeline: every feature is a 90-day trailing aggregate or a static content attribute, all knowable before the current-period decision.
- Label-derived / sibling columns: trend_direction and trend_pct are the label's own source column and are excluded from every feature list — confirmed by name below, not just by memory.
- Product flags: none of FlyRank's internal optimization flags are in the feature set.
- Grouped split: confirmed in Section 2 — 0 client overlap between train and test.
- Base rate printed next to every metric: done in Section 2 (0.511 / 0.542).
- Feature importance sanity-check: below, I deliberately re-introduce trend_pct (the label's direct source) as a feature to confirm my test harness actually catches a leak when one exists — then remove it again.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
leak_suspects = ["trend_direction", "trend_pct"]
in_features = [c for c in leak_suspects if c in MODEL_NUMERIC + MODEL_CAT]
print(f"Label-derived columns present in real feature set: {in_features} (should be empty list)")

X_leak = X.copy()
X_leak["trend_pct"] = df["trend_pct"].values

Xtr_l, Xte_l = X_leak.iloc[train_idx], X_leak.iloc[test_idx]
auc_leak, p50_leak, _ = run_rf(Xtr_l, Xte_l, y.iloc[train_idx], y.iloc[test_idx])
print(f"\nWITHOUT leak (real model, grouped split): ROC AUC = {auc_after:.3f}")
print(f"WITH deliberate trend_pct leak (same split): ROC AUC = {auc_leak:.3f}  <- confession: jumps toward 1.0")

rf_leak = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
                                  n_estimators=200, n_jobs=-1, random_state=42).fit(Xtr_l, y.iloc[train_idx])
importances = pd.Series(rf_leak.feature_importances_, index=X_leak.columns).sort_values(ascending=False)
print("\nTop 5 feature importances WITH the leak:")
print(importances.head(5).round(4))
print(f"\n=> My test harness correctly flags a real leak. My actual feature set (Section 2) has none of this.")

Label-derived columns present in real feature set: [] (should be empty list)

WITHOUT leak (real model, grouped split): ROC AUC = 0.610
WITH deliberate trend_pct leak (same split): ROC AUC = 1.000  <- confession: jumps toward 1.0

Top 5 feature importances WITH the leak:
trend_pct                0.7820
days_with_impressions    0.0367
log_impressions_90d      0.0299
avg_position             0.0217
content_age_days         0.0192
dtype: float64

=> My test harness correctly flags a real leak. My actual feature set (Section 2) has none of this.


## 4. Claim rewrite
My boldest sentence from earlier work (Week-5 model report / ML-08): "Random Forest beats the rule-based baseline for flagging declining content."

That claim was true only under the naive random split. Under the honest, client-grouped split, Precision@50 for random forest (~0.54) is close to — and on this run, not clearly above — the rule baseline (~0.62) and the base rate (~0.51).

Rewritten, safe version: "On this portfolio's held-out clients, the Random Forest score does not show a measured advantage over the Week-4 rule baseline at Precision@50; both sit close to the base rate. The model's random-split number (Precision@50 ≈ 0.90) is not trustworthy — it reflects client memorization, not generalizable skill. The honest, directional takeaway is that neither the rule nor the model is a strong out-of-sample decliner-flagger yet on this feature set; either can serve as a decision-support starting point for manual review, not an automatic ranking a team should trust unreviewed."

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
bold_claim = "Random Forest beats the rule-based baseline for flagging declining content."
safe_claim = (
    "On held-out clients, the Random Forest score shows no measured advantage over the "
    "rule baseline at Precision@50 (both near the base rate); the earlier higher number came "
    "from a random split that let the model see each client in both training and test."
)
print("BEFORE (bold):", bold_claim)
print("\nAFTER (safe): ", safe_claim)
print(f"\nEvidence check -- grouped split numbers this notebook actually produced:")
print(result.to_string(index=False))

BEFORE (bold): Random Forest beats the rule-based baseline for flagging declining content.

AFTER (safe):  On held-out clients, the Random Forest score shows no measured advantage over the rule baseline at Precision@50 (both near the base rate); the earlier higher number came from a random split that let the model see each client in both training and test.

Evidence check -- grouped split numbers this notebook actually produced:
                       split  roc_auc  precision_at_50  base_rate
   BEFORE - random row split    0.758             0.90      0.542
AFTER - client-grouped split    0.610             0.54      0.511


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.